# WiDS Global Datathon 2026 — Internal Survival Superblend


In [1]:
import os
import gc
import json
import math
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import lightgbm as lgb
from catboost import CatBoostClassifier, CatBoostRegressor

SEEDS = [11, 23, 37, 71, 101]
N_SPLITS = 5
HORIZONS = [12, 24, 48]

WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
OUTPUT_PATH = os.path.join(WORK_DIR, "submission.csv")
OOF_PATH = os.path.join(WORK_DIR, "oof_predictions.csv")
WEIGHTS_PATH = os.path.join(WORK_DIR, "blend_weights.json")


def find_data_dir():
    direct_candidates = [
        "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/widsworldwide-globaldathon26",
        "/kaggle/input/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/wiDSworldwide_globaldathon26",
        ".",
    ]
    needed = {"train.csv", "test.csv", "sample_submission.csv"}
    for path in direct_candidates:
        if os.path.isdir(path):
            files = set(os.listdir(path))
            if needed.issubset(files):
                return path
    kaggle_input = "/kaggle/input"
    if os.path.isdir(kaggle_input):
        for root, _, files in os.walk(kaggle_input):
            if needed.issubset(set(files)):
                return root
    raise FileNotFoundError("Could not locate train.csv / test.csv / sample_submission.csv")


DATA_DIR = find_data_dir()

train_raw = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_raw = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

TIME = train_raw["time_to_hit_hours"].to_numpy(dtype=float)
EVENT = train_raw["event"].to_numpy(dtype=int)

RAW_FEATURES = [c for c in train_raw.columns if c not in ["event_id", "time_to_hit_hours", "event"]]


def pct_rank(values, ref_values):
    ref = np.sort(np.asarray(ref_values, dtype=float))
    if ref.size == 0:
        return np.zeros(len(values), dtype=float)
    return np.searchsorted(ref, np.asarray(values, dtype=float), side="right") / ref.size


def core_signals(df):
    d = df.copy()

    dist_m = d["dist_min_ci_0_5h"].astype(float).clip(lower=1.0)
    dist_km = dist_m / 1000.0

    close_speed = d["closing_speed_m_per_h"].astype(float).clip(lower=0.0)
    along_pos = d["along_track_speed"].astype(float).clip(lower=0.0)
    radial_pos = d["radial_growth_rate_m_per_h"].astype(float).clip(lower=0.0)
    projected_pos = d["projected_advance_m"].astype(float).clip(lower=0.0)
    align = d["alignment_abs"].astype(float).clip(lower=0.0, upper=1.0)

    area_first = d["area_first_ha"].astype(float).clip(lower=0.0)
    growth_abs = d["area_growth_abs_0_5h"].astype(float).clip(lower=0.0)
    radial_growth = d["radial_growth_m"].astype(float).clip(lower=0.0)

    reliability = (1.0 - 0.55 * d["low_temporal_resolution_0_5h"].astype(float))
    reliability = reliability * np.clip(d["dt_first_last_0_5h"].astype(float) / 5.0, 0.2, 1.0)
    reliability = reliability * np.clip((d["num_perimeters_0_5h"].astype(float) - 0.5) / 3.5, 0.35, 1.0)
    reliability = reliability * np.clip(0.35 + 0.65 * d["dist_fit_r2_0_5h"].astype(float), 0.2, 1.0)

    fire_radius_m = np.sqrt(np.maximum(area_first, 0.0) * 10000.0 / np.pi)
    eff1 = close_speed + 0.70 * radial_pos + 0.35 * along_pos
    eff2 = np.maximum(close_speed, along_pos) + radial_pos

    margin_m = dist_m - fire_radius_m - projected_pos

    wave_eta = np.where(eff1 > 1.0, np.maximum(margin_m, 0.0) / eff1, 1e6)
    close_eta = np.where(close_speed > 1.0, dist_m / close_speed, 1e6)
    combo_eta = np.minimum(wave_eta, close_eta)

    threat_momentum = align * eff1 / np.log1p(dist_m)
    threat_gravity = (
        np.log1p(growth_abs) + np.log1p(radial_growth) + 0.5 * np.log1p(area_first)
    ) * (0.25 + align) / (dist_km + 0.25)

    return {
        "dist_m": dist_m,
        "dist_km": dist_km,
        "close_speed": close_speed,
        "along_pos": along_pos,
        "radial_pos": radial_pos,
        "projected_pos": projected_pos,
        "align": align,
        "area_first": area_first,
        "growth_abs": growth_abs,
        "radial_growth": radial_growth,
        "reliability": reliability,
        "fire_radius_m": fire_radius_m,
        "eff1": eff1,
        "eff2": eff2,
        "margin_m": margin_m,
        "wave_eta": wave_eta,
        "close_eta": close_eta,
        "combo_eta": combo_eta,
        "threat_momentum": threat_momentum,
        "threat_gravity": threat_gravity,
    }


def add_features(df, fit_ref=None):
    ref = fit_ref if fit_ref is not None else df

    cur = df.copy()
    sig = core_signals(cur)
    ref_sig = core_signals(ref)

    cur["dist_km"] = sig["dist_km"]
    cur["log_distance"] = np.log1p(sig["dist_m"])
    cur["inv_distance"] = 1.0 / (sig["dist_km"] + 0.1)
    cur["inv_distance_sq"] = cur["inv_distance"] ** 2
    cur["sqrt_distance"] = np.sqrt(sig["dist_m"])

    cur["fire_radius_km"] = sig["fire_radius_m"] / 1000.0
    cur["near_miss_margin_km"] = sig["margin_m"] / 1000.0
    cur["radius_to_distance"] = sig["fire_radius_m"] / sig["dist_m"]

    cur["effective_close_1"] = sig["eff1"]
    cur["effective_close_2"] = sig["eff2"]
    cur["trusted_close_1"] = sig["eff1"] * sig["reliability"]
    cur["trusted_close_2"] = sig["eff2"] * sig["reliability"]

    cur["wavefront_eta_h"] = np.clip(sig["wave_eta"], 0.0, 1e6)
    cur["closing_eta_h"] = np.clip(sig["close_eta"], 0.0, 1e6)
    cur["eta_combo_h"] = np.clip(sig["combo_eta"], 0.0, 1e6)
    cur["log_wavefront_eta"] = np.log1p(cur["wavefront_eta_h"])
    cur["log_eta_combo"] = np.log1p(cur["eta_combo_h"])

    cur["threat_momentum"] = sig["threat_momentum"]
    cur["threat_gravity"] = sig["threat_gravity"]
    cur["threat_rel"] = sig["threat_momentum"] * sig["reliability"]
    cur["reliability"] = sig["reliability"]

    cur["projected_advance_pos"] = sig["projected_pos"]
    cur["closing_distance_pos"] = (-cur["dist_change_ci_0_5h"].astype(float)).clip(lower=0.0)
    cur["slope_toward"] = (-cur["dist_slope_ci_0_5h"].astype(float)).clip(lower=0.0)
    cur["accel_toward"] = (-cur["dist_accel_m_per_h2"].astype(float)).clip(lower=0.0)

    cur["fire_urgency"] = sig["eff1"] * cur["num_perimeters_0_5h"].astype(float) * (0.5 + sig["align"])
    cur["growth_to_distance"] = np.log1p(sig["growth_abs"]) / (sig["dist_km"] + 0.2)
    cur["area_to_distance"] = np.log1p(sig["area_first"]) / (sig["dist_km"] + 0.2)

    cur["zone_2km"] = (sig["dist_km"] < 2.0).astype(float)
    cur["zone_5km"] = (sig["dist_km"] < 5.0).astype(float)
    cur["zone_8km"] = (sig["dist_km"] < 8.0).astype(float)
    cur["zone_15km"] = (sig["dist_km"] < 15.0).astype(float)
    cur["zone_30km"] = (sig["dist_km"] < 30.0).astype(float)

    cur["hour_sin"] = np.sin(2.0 * np.pi * cur["event_start_hour"].astype(float) / 24.0)
    cur["hour_cos"] = np.cos(2.0 * np.pi * cur["event_start_hour"].astype(float) / 24.0)
    cur["month_sin"] = np.sin(2.0 * np.pi * (cur["event_start_month"].astype(float) - 1.0) / 12.0)
    cur["month_cos"] = np.cos(2.0 * np.pi * (cur["event_start_month"].astype(float) - 1.0) / 12.0)
    cur["dow_sin"] = np.sin(2.0 * np.pi * cur["event_start_dayofweek"].astype(float) / 7.0)
    cur["dow_cos"] = np.cos(2.0 * np.pi * cur["event_start_dayofweek"].astype(float) / 7.0)

    cur["distance_rank"] = pct_rank(sig["dist_m"], ref_sig["dist_m"])
    cur["eta_rank"] = pct_rank(sig["combo_eta"], ref_sig["combo_eta"])
    cur["threat_rank"] = pct_rank(sig["threat_gravity"], ref_sig["threat_gravity"])

    drop_cols = [c for c in ["relative_growth_0_5h", "spread_bearing_deg"] if c in cur.columns]
    cur = cur.drop(columns=drop_cols)
    cur = cur.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return cur


train_fe = add_features(train_raw[RAW_FEATURES], fit_ref=train_raw[RAW_FEATURES])
test_fe = add_features(test_raw[RAW_FEATURES], fit_ref=train_raw[RAW_FEATURES])

CAT_FEATURES = list(train_fe.columns)

LGB_FEATURES = [
    "dist_min_ci_0_5h",
    "dist_km",
    "log_distance",
    "inv_distance",
    "fire_radius_km",
    "near_miss_margin_km",
    "effective_close_1",
    "trusted_close_1",
    "alignment_abs",
    "threat_rel",
    "growth_to_distance",
    "area_to_distance",
    "num_perimeters_0_5h",
    "dt_first_last_0_5h",
    "low_temporal_resolution_0_5h",
    "distance_rank",
    "eta_rank",
    "threat_rank",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
    "projected_advance_pos",
    "slope_toward",
    "accel_toward",
    "dist_fit_r2_0_5h",
    "zone_5km",
    "zone_8km",
    "zone_15km",
]
LGB_FEATURES = [c for c in LGB_FEATURES if c in train_fe.columns]

LINEAR_FEATURES = [
    "dist_km",
    "log_distance",
    "inv_distance",
    "fire_radius_km",
    "near_miss_margin_km",
    "trusted_close_1",
    "alignment_abs",
    "threat_rel",
    "growth_to_distance",
    "area_to_distance",
    "num_perimeters_0_5h",
    "dt_first_last_0_5h",
    "low_temporal_resolution_0_5h",
    "distance_rank",
    "eta_rank",
    "threat_rank",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
]
LINEAR_FEATURES = [c for c in LINEAR_FEATURES if c in train_fe.columns]

X_cat_train = train_fe[CAT_FEATURES].copy()
X_cat_test = test_fe[CAT_FEATURES].copy()
X_lgb_train = train_fe[LGB_FEATURES].copy()
X_lgb_test = test_fe[LGB_FEATURES].copy()
X_lin_train = train_fe[LINEAR_FEATURES].copy()
X_lin_test = test_fe[LINEAR_FEATURES].copy()


def make_full_strata():
    dist_bin = pd.qcut(
        train_fe["dist_km"].rank(method="first"),
        q=3,
        labels=False,
        duplicates="drop",
    ).astype(int).to_numpy()
    return EVENT.astype(int) * 10 + dist_bin.astype(int)


FULL_STRATA = make_full_strata()


def make_binary_target(time_values, event_values, horizon):
    y = ((event_values == 1) & (time_values <= horizon)).astype(int)
    mask = ~((event_values == 0) & (time_values < horizon))
    return y, mask


def compute_brier(time_values, event_values, prob, horizon):
    valid = ~((event_values == 0) & (time_values < horizon))
    y_true = ((event_values == 1) & (time_values <= horizon)).astype(float)
    p = np.clip(np.asarray(prob, dtype=float), 0.0, 1.0)
    if valid.sum() == 0:
        return 0.0
    return float(np.mean((p[valid] - y_true[valid]) ** 2))


def compute_c_index(time_values, event_values, risk):
    time_values = np.asarray(time_values, dtype=float)
    event_values = np.asarray(event_values, dtype=int)
    risk = np.asarray(risk, dtype=float)

    concordant = 0.0
    comparable = 0.0
    n = len(time_values)

    for i in range(n):
        if event_values[i] != 1:
            continue
        for j in range(n):
            if i == j or time_values[i] >= time_values[j]:
                continue
            comparable += 1.0
            if risk[i] > risk[j]:
                concordant += 1.0
            elif risk[i] == risk[j]:
                concordant += 0.5

    return float(concordant / comparable) if comparable > 0 else 0.5


def compute_proxy_hybrid(pred_12, pred_24, pred_48, pred_72):
    risk = 0.30 * np.asarray(pred_24, dtype=float) + 0.40 * np.asarray(pred_48, dtype=float) + 0.30 * np.asarray(pred_72, dtype=float)
    c_idx = compute_c_index(TIME, EVENT, risk)
    b24 = compute_brier(TIME, EVENT, pred_24, 24)
    b48 = compute_brier(TIME, EVENT, pred_48, 48)
    b72 = compute_brier(TIME, EVENT, pred_72, 72)
    wb = 0.30 * b24 + 0.40 * b48 + 0.30 * b72
    score = 0.30 * c_idx + 0.70 * (1.0 - wb)
    return score, c_idx, wb, b24, b48, b72


def monotone_rows(arr):
    out = np.clip(np.asarray(arr, dtype=float), 0.0, 1.0).copy()
    for j in range(1, out.shape[1]):
        out[:, j] = np.maximum(out[:, j], out[:, j - 1])
    return np.clip(out, 0.0, 1.0)


def km_event_cdf(time_values, event_values, horizon):
    t = np.asarray(time_values, dtype=float)
    e = np.asarray(event_values, dtype=int)

    surv = 1.0
    unique_times = np.sort(np.unique(t[t <= horizon]))
    for tau in unique_times:
        at_risk = np.sum(t >= tau)
        d = np.sum((t == tau) & (e == 1))
        if at_risk > 0 and d > 0:
            surv *= (1.0 - d / at_risk)
    return float(1.0 - surv)


def pseudo_values(time_values, event_values, horizon):
    t = np.asarray(time_values, dtype=float)
    e = np.asarray(event_values, dtype=int)
    n = len(t)

    full = km_event_cdf(t, e, horizon)
    out = np.zeros(n, dtype=float)

    for i in range(n):
        loo_t = np.delete(t, i)
        loo_e = np.delete(e, i)
        loo = km_event_cdf(loo_t, loo_e, horizon)
        out[i] = n * full - (n - 1) * loo

    return np.clip(out, -0.25, 1.25)


def build_urgency_target(time_values, event_values):
    pv12 = pseudo_values(time_values, event_values, 12)
    pv24 = pseudo_values(time_values, event_values, 24)
    pv48 = pseudo_values(time_values, event_values, 48)
    timing_bonus = ((event_values == 1).astype(float) * (1.0 - np.minimum(time_values, 72.0) / 72.0))
    out = 0.15 * pv12 + 0.35 * pv24 + 0.50 * pv48 + 0.15 * timing_bonus
    return np.clip(out, -0.25, 1.50)


def censor_survival_at(time_values, event_values, horizon):
    t = np.asarray(time_values, dtype=float)
    e = np.asarray(event_values, dtype=int)

    surv = 1.0
    unique_times = np.sort(np.unique(t[t <= horizon]))
    for tau in unique_times:
        at_risk = np.sum(t >= tau)
        d_cens = np.sum((t == tau) & (e == 0))
        if at_risk > 0 and d_cens > 0:
            surv *= (1.0 - d_cens / at_risk)
    return float(max(surv, 1e-3))


def ipcw_weights(time_values, event_values, horizon):
    t = np.asarray(time_values, dtype=float)
    e = np.asarray(event_values, dtype=int)
    w = np.zeros(len(t), dtype=float)

    for i in range(len(t)):
        if e[i] == 1 and t[i] <= horizon:
            w[i] = 1.0 / censor_survival_at(t, e, t[i])
        else:
            w[i] = 1.0 / censor_survival_at(t, e, horizon)

    return np.clip(w, 0.25, 20.0)


def fit_model(model, X, y, sample_weight=None):
    if sample_weight is None:
        model.fit(X, y)
        return model

    if isinstance(model, Pipeline):
        last_name = model.steps[-1][0]
        try:
            model.fit(X, y, **{f"{last_name}__sample_weight": sample_weight})
            return model
        except Exception:
            model.fit(X, y)
            return model

    try:
        model.fit(X, y, sample_weight=sample_weight)
    except Exception:
        model.fit(X, y)
    return model


def predict_binary(model, X):
    pred = model.predict_proba(X)
    pred = np.asarray(pred)
    if pred.ndim == 2:
        return pred[:, 1]
    return pred


def repeated_isotonic(signal_train, signal_test, horizon, seeds=SEEDS, n_splits=N_SPLITS):
    signal_train = np.asarray(signal_train, dtype=float)
    signal_test = np.asarray(signal_test, dtype=float)

    y, mask = make_binary_target(TIME, EVENT, horizon)
    valid_idx = np.where(mask)[0]
    y_valid = y[valid_idx]

    oof = np.zeros(len(signal_train), dtype=float)
    cnt = np.zeros(len(signal_train), dtype=float)
    full_fit = np.zeros(len(signal_train), dtype=float)
    test_pred = np.zeros(len(signal_test), dtype=float)

    for seed in seeds:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_sub, va_sub in cv.split(valid_idx, y_valid):
            tr_idx = valid_idx[tr_sub]
            va_idx = valid_idx[va_sub]

            ir = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip")
            try:
                ir.fit(signal_train[tr_idx], y[tr_idx])
                fold_pred = ir.predict(signal_train[va_idx])
            except Exception:
                fold_pred = np.full(len(va_idx), float(np.mean(y[tr_idx])))

            oof[va_idx] += fold_pred
            cnt[va_idx] += 1.0

        ir_full = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip")
        try:
            ir_full.fit(signal_train[valid_idx], y[valid_idx])
            full_fit += ir_full.predict(signal_train)
            test_pred += ir_full.predict(signal_test)
        except Exception:
            mean_val = float(np.mean(y[valid_idx]))
            full_fit += mean_val
            test_pred += mean_val

    full_fit /= len(seeds)
    test_pred /= len(seeds)
    nz = cnt > 0
    oof[nz] /= cnt[nz]
    oof[~nz] = full_fit[~nz]
    return np.clip(oof, 0.0, 1.0), np.clip(test_pred, 0.0, 1.0)


def repeated_binary_cv(X_train, X_test, horizon, build_model, use_ipcw=True, seeds=SEEDS, n_splits=N_SPLITS):
    y, mask = make_binary_target(TIME, EVENT, horizon)
    valid_idx = np.where(mask)[0]
    y_valid = y[valid_idx]

    oof = np.zeros(len(X_train), dtype=float)
    cnt = np.zeros(len(X_train), dtype=float)
    full_fit = np.zeros(len(X_train), dtype=float)
    test_pred = np.zeros(len(X_test), dtype=float)

    for seed in seeds:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_sub, va_sub in cv.split(valid_idx, y_valid):
            tr_idx = valid_idx[tr_sub]
            va_idx = valid_idx[va_sub]

            model = build_model(seed, horizon)
            sw = ipcw_weights(TIME[tr_idx], EVENT[tr_idx], horizon) if use_ipcw else None
            fit_model(model, X_train.iloc[tr_idx], y[tr_idx], sample_weight=sw)

            oof[va_idx] += predict_binary(model, X_train.iloc[va_idx])
            cnt[va_idx] += 1.0

        model_full = build_model(seed, horizon)
        sw_full = ipcw_weights(TIME[valid_idx], EVENT[valid_idx], horizon) if use_ipcw else None
        fit_model(model_full, X_train.iloc[valid_idx], y[valid_idx], sample_weight=sw_full)

        full_fit += predict_binary(model_full, X_train)
        test_pred += predict_binary(model_full, X_test)

    full_fit /= len(seeds)
    test_pred /= len(seeds)
    nz = cnt > 0
    oof[nz] /= cnt[nz]
    oof[~nz] = full_fit[~nz]
    return np.clip(oof, 0.0, 1.0), np.clip(test_pred, 0.0, 1.0)


def repeated_pseudo_regression(X_train, X_test, horizon, build_model, seeds=SEEDS, n_splits=N_SPLITS):
    oof = np.zeros(len(X_train), dtype=float)
    cnt = np.zeros(len(X_train), dtype=float)
    full_fit = np.zeros(len(X_train), dtype=float)
    test_pred = np.zeros(len(X_test), dtype=float)

    target_full = pseudo_values(TIME, EVENT, horizon)

    for seed in seeds:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_idx, va_idx in cv.split(X_train, FULL_STRATA):
            target_tr = pseudo_values(TIME[tr_idx], EVENT[tr_idx], horizon)
            model = build_model(seed, horizon)
            model.fit(X_train.iloc[tr_idx], target_tr)
            oof[va_idx] += np.asarray(model.predict(X_train.iloc[va_idx]), dtype=float)
            cnt[va_idx] += 1.0

        model_full = build_model(seed, horizon)
        model_full.fit(X_train, target_full)
        full_fit += np.asarray(model_full.predict(X_train), dtype=float)
        test_pred += np.asarray(model_full.predict(X_test), dtype=float)

    full_fit /= len(seeds)
    test_pred /= len(seeds)
    nz = cnt > 0
    oof[nz] /= cnt[nz]
    oof[~nz] = full_fit[~nz]
    return oof, test_pred


def repeated_urgency_regression(X_train, X_test, build_model, seeds=SEEDS, n_splits=N_SPLITS):
    oof = np.zeros(len(X_train), dtype=float)
    cnt = np.zeros(len(X_train), dtype=float)
    full_fit = np.zeros(len(X_train), dtype=float)
    test_pred = np.zeros(len(X_test), dtype=float)

    target_full = build_urgency_target(TIME, EVENT)

    for seed in seeds:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_idx, va_idx in cv.split(X_train, FULL_STRATA):
            target_tr = build_urgency_target(TIME[tr_idx], EVENT[tr_idx])
            model = build_model(seed)
            model.fit(X_train.iloc[tr_idx], target_tr)
            oof[va_idx] += np.asarray(model.predict(X_train.iloc[va_idx]), dtype=float)
            cnt[va_idx] += 1.0

        model_full = build_model(seed)
        model_full.fit(X_train, target_full)
        full_fit += np.asarray(model_full.predict(X_train), dtype=float)
        test_pred += np.asarray(model_full.predict(X_test), dtype=float)

    full_fit /= len(seeds)
    test_pred /= len(seeds)
    nz = cnt > 0
    oof[nz] /= cnt[nz]
    oof[~nz] = full_fit[~nz]
    return oof, test_pred


def soft_gate(dist_km, center, width):
    z = np.clip((np.asarray(dist_km, dtype=float) - center) / max(width, 1e-6), -60.0, 60.0)
    return 1.0 / (1.0 + np.exp(z))


def prior_vector(names, prior_map):
    vec = np.array([prior_map.get(name, 0.0) for name in names], dtype=float)
    vec = np.clip(vec, 0.0, None)
    if vec.sum() <= 0:
        vec = np.ones(len(names), dtype=float)
    return vec / vec.sum()


def optimize_simplex(P, y, prior, reg, shrink):
    P = np.asarray(P, dtype=float)
    y = np.asarray(y, dtype=float)
    prior = np.asarray(prior, dtype=float)

    if P.ndim == 1:
        P = P.reshape(-1, 1)
    if P.shape[1] == 1:
        return np.array([1.0], dtype=float)

    P = np.clip(P, 0.0, 1.0)

    def objective(w):
        pred = P @ w
        return float(np.mean((pred - y) ** 2) + reg * np.sum((w - prior) ** 2))

    bounds = [(0.0, 1.0)] * P.shape[1]
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]

    try:
        result = minimize(
            objective,
            prior.copy(),
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={"maxiter": 500, "ftol": 1e-10},
        )
        if result.success:
            w = np.clip(result.x, 0.0, None)
            w = w / w.sum()
        else:
            w = prior.copy()
    except Exception:
        w = prior.copy()

    w = shrink * w + (1.0 - shrink) * prior
    w = np.clip(w, 0.0, None)
    w = w / w.sum()
    return w


print("DATA_DIR:", DATA_DIR)
print("TRAIN:", train_raw.shape, "TEST:", test_raw.shape)

anchor_signal_train = (
    1.0 / (np.maximum(train_fe["near_miss_margin_km"].to_numpy(dtype=float), 0.0) + 0.20)
    + 0.10 * train_fe["threat_rank"].to_numpy(dtype=float)
)
anchor_signal_test = (
    1.0 / (np.maximum(test_fe["near_miss_margin_km"].to_numpy(dtype=float), 0.0) + 0.20)
    + 0.10 * test_fe["threat_rank"].to_numpy(dtype=float)
)

eta_signal_train = (
    -train_fe["log_eta_combo"].to_numpy(dtype=float)
    + 0.20 * train_fe["threat_rank"].to_numpy(dtype=float)
    + 0.10 * train_fe["reliability"].to_numpy(dtype=float)
)
eta_signal_test = (
    -test_fe["log_eta_combo"].to_numpy(dtype=float)
    + 0.20 * test_fe["threat_rank"].to_numpy(dtype=float)
    + 0.10 * test_fe["reliability"].to_numpy(dtype=float)
)

anchor_oof = {}
anchor_test = {}
eta_oof = {}
eta_test = {}

print("Calibrating anchor / physics signals...")
for h in HORIZONS:
    anchor_oof[h], anchor_test[h] = repeated_isotonic(anchor_signal_train, anchor_signal_test, h)
    eta_oof[h], eta_test[h] = repeated_isotonic(eta_signal_train, eta_signal_test, h)


def build_cat_classifier(seed, horizon):
    params = {
        12: {"iterations": 420, "depth": 4, "learning_rate": 0.030, "l2_leaf_reg": 10.0},
        24: {"iterations": 480, "depth": 4, "learning_rate": 0.028, "l2_leaf_reg": 12.0},
        48: {"iterations": 540, "depth": 4, "learning_rate": 0.026, "l2_leaf_reg": 14.0},
    }[horizon]
    return CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="Logloss",
        bootstrap_type="Bernoulli",
        subsample=0.85,
        random_strength=1.0,
        allow_writing_files=False,
        verbose=False,
        thread_count=-1,
        random_seed=seed,
        **params,
    )


def build_lgb_classifier(seed, horizon):
    params = {
        12: {"n_estimators": 280, "learning_rate": 0.035, "max_depth": 3, "num_leaves": 7, "subsample": 0.85, "colsample_bytree": 0.80, "min_child_samples": 8, "reg_alpha": 1.0, "reg_lambda": 3.0},
        24: {"n_estimators": 320, "learning_rate": 0.032, "max_depth": 3, "num_leaves": 7, "subsample": 0.85, "colsample_bytree": 0.80, "min_child_samples": 10, "reg_alpha": 1.0, "reg_lambda": 3.5},
        48: {"n_estimators": 360, "learning_rate": 0.030, "max_depth": 3, "num_leaves": 8, "subsample": 0.88, "colsample_bytree": 0.82, "min_child_samples": 12, "reg_alpha": 1.2, "reg_lambda": 4.0},
    }[horizon]
    return lgb.LGBMClassifier(
        objective="binary",
        random_state=seed,
        n_jobs=-1,
        verbose=-1,
        **params,
    )


def build_logit(seed, horizon):
    c_map = {12: 0.60, 24: 0.85, 48: 1.10}
    return Pipeline(
        [
            ("scale", StandardScaler()),
            ("logit", LogisticRegression(C=c_map[horizon], max_iter=5000, solver="lbfgs")),
        ]
    )


def build_pseudo_cat(seed, horizon):
    params = {
        12: {"iterations": 460, "depth": 4, "learning_rate": 0.028, "l2_leaf_reg": 10.0},
        24: {"iterations": 520, "depth": 4, "learning_rate": 0.026, "l2_leaf_reg": 12.0},
        48: {"iterations": 580, "depth": 4, "learning_rate": 0.024, "l2_leaf_reg": 14.0},
    }[horizon]
    return CatBoostRegressor(
        loss_function="RMSE",
        bootstrap_type="Bernoulli",
        subsample=0.85,
        random_strength=1.0,
        allow_writing_files=False,
        verbose=False,
        thread_count=-1,
        random_seed=seed,
        **params,
    )


def build_pseudo_et(seed, horizon):
    depth_map = {12: 5, 24: 6, 48: 6}
    return ExtraTreesRegressor(
        n_estimators=500,
        max_depth=depth_map[horizon],
        min_samples_leaf=4,
        max_features="sqrt",
        random_state=seed,
        n_jobs=-1,
    )


def build_urgency_cat(seed):
    return CatBoostRegressor(
        loss_function="RMSE",
        iterations=520,
        depth=4,
        learning_rate=0.026,
        l2_leaf_reg=12.0,
        bootstrap_type="Bernoulli",
        subsample=0.85,
        random_strength=1.0,
        allow_writing_files=False,
        verbose=False,
        thread_count=-1,
        random_seed=seed,
    )


print("Training direct horizon models...")
cat_oof = {}
cat_test = {}
lgb_oof = {}
lgb_test = {}
logit_oof = {}
logit_test = {}

for h in HORIZONS:
    cat_oof[h], cat_test[h] = repeated_binary_cv(X_cat_train, X_cat_test, h, build_cat_classifier, use_ipcw=True)
    lgb_oof[h], lgb_test[h] = repeated_binary_cv(X_lgb_train, X_lgb_test, h, build_lgb_classifier, use_ipcw=True)
    logit_oof[h], logit_test[h] = repeated_binary_cv(X_lin_train, X_lin_test, h, build_logit, use_ipcw=True)

    cat_oof[h] = np.clip(0.92 * cat_oof[h] + 0.08 * anchor_oof[h], 0.0, 1.0)
    cat_test[h] = np.clip(0.92 * cat_test[h] + 0.08 * anchor_test[h], 0.0, 1.0)

    lgb_oof[h] = np.clip(0.90 * lgb_oof[h] + 0.10 * anchor_oof[h], 0.0, 1.0)
    lgb_test[h] = np.clip(0.90 * lgb_test[h] + 0.10 * anchor_test[h], 0.0, 1.0)

    logit_oof[h] = np.clip(0.85 * logit_oof[h] + 0.15 * anchor_oof[h], 0.0, 1.0)
    logit_test[h] = np.clip(0.85 * logit_test[h] + 0.15 * anchor_test[h], 0.0, 1.0)


print("Training pseudo-value regressors...")
pseudo_cat_raw_oof = {}
pseudo_cat_raw_test = {}
pseudo_et_raw_oof = {}
pseudo_et_raw_test = {}

for h in HORIZONS:
    pseudo_cat_raw_oof[h], pseudo_cat_raw_test[h] = repeated_pseudo_regression(X_cat_train, X_cat_test, h, build_pseudo_cat)
    pseudo_et_raw_oof[h], pseudo_et_raw_test[h] = repeated_pseudo_regression(X_lgb_train, X_lgb_test, h, build_pseudo_et)

pseudo_cat_oof = {}
pseudo_cat_test = {}
pseudo_et_oof = {}
pseudo_et_test = {}

for h in HORIZONS:
    pseudo_cat_oof[h], pseudo_cat_test[h] = repeated_isotonic(pseudo_cat_raw_oof[h], pseudo_cat_raw_test[h], h)
    pseudo_et_oof[h], pseudo_et_test[h] = repeated_isotonic(pseudo_et_raw_oof[h], pseudo_et_raw_test[h], h)

    pseudo_cat_oof[h] = np.clip(0.92 * pseudo_cat_oof[h] + 0.08 * anchor_oof[h], 0.0, 1.0)
    pseudo_cat_test[h] = np.clip(0.92 * pseudo_cat_test[h] + 0.08 * anchor_test[h], 0.0, 1.0)

    pseudo_et_oof[h] = np.clip(0.90 * pseudo_et_oof[h] + 0.10 * anchor_oof[h], 0.0, 1.0)
    pseudo_et_test[h] = np.clip(0.90 * pseudo_et_test[h] + 0.10 * anchor_test[h], 0.0, 1.0)


print("Training shared urgency model...")
urgency_raw_oof, urgency_raw_test = repeated_urgency_regression(X_cat_train, X_cat_test, build_urgency_cat)
urgency_oof = {}
urgency_test = {}
for h in HORIZONS:
    urgency_oof[h], urgency_test[h] = repeated_isotonic(urgency_raw_oof, urgency_raw_test, h)
    urgency_oof[h] = np.clip(0.90 * urgency_oof[h] + 0.10 * anchor_oof[h], 0.0, 1.0)
    urgency_test[h] = np.clip(0.90 * urgency_test[h] + 0.10 * anchor_test[h], 0.0, 1.0)

for h in HORIZONS:
    eta_oof[h] = np.clip(0.85 * eta_oof[h] + 0.15 * anchor_oof[h], 0.0, 1.0)
    eta_test[h] = np.clip(0.85 * eta_test[h] + 0.15 * anchor_test[h], 0.0, 1.0)


OOF_BASES = {
    "cat": cat_oof,
    "lgb": lgb_oof,
    "logit": logit_oof,
    "pseudo_cat": pseudo_cat_oof,
    "pseudo_et": pseudo_et_oof,
    "urgency": urgency_oof,
    "eta": eta_oof,
    "anchor": anchor_oof,
}

TEST_BASES = {
    "cat": cat_test,
    "lgb": lgb_test,
    "logit": logit_test,
    "pseudo_cat": pseudo_cat_test,
    "pseudo_et": pseudo_et_test,
    "urgency": urgency_test,
    "eta": eta_test,
    "anchor": anchor_test,
}

PRIORS = {
    12: {
        "near": {"cat": 0.24, "lgb": 0.14, "logit": 0.10, "pseudo_cat": 0.08, "pseudo_et": 0.04, "urgency": 0.10, "eta": 0.24, "anchor": 0.06},
        "far": {"cat": 0.16, "lgb": 0.12, "logit": 0.10, "pseudo_cat": 0.22, "pseudo_et": 0.10, "urgency": 0.20, "eta": 0.04, "anchor": 0.06},
    },
    24: {
        "near": {"cat": 0.22, "lgb": 0.14, "logit": 0.08, "pseudo_cat": 0.12, "pseudo_et": 0.05, "urgency": 0.16, "eta": 0.18, "anchor": 0.05},
        "far": {"cat": 0.14, "lgb": 0.12, "logit": 0.08, "pseudo_cat": 0.25, "pseudo_et": 0.10, "urgency": 0.22, "eta": 0.03, "anchor": 0.06},
    },
    48: {
        "near": {"cat": 0.18, "lgb": 0.10, "logit": 0.07, "pseudo_cat": 0.18, "pseudo_et": 0.07, "urgency": 0.20, "eta": 0.12, "anchor": 0.08},
        "far": {"cat": 0.10, "lgb": 0.10, "logit": 0.06, "pseudo_cat": 0.30, "pseudo_et": 0.12, "urgency": 0.24, "eta": 0.02, "anchor": 0.06},
    },
}

BLEND_CFG = {
    12: {"reg": 0.08, "shrink": 0.25},
    24: {"reg": 0.07, "shrink": 0.22},
    48: {"reg": 0.06, "shrink": 0.18},
}

dist_train = train_fe["dist_km"].to_numpy(dtype=float)
dist_test = test_fe["dist_km"].to_numpy(dtype=float)

weight_log = {}
blend_oof = {}
blend_test = {}

print("Blending internal model families...")

for h in HORIZONS:
    y, mask = make_binary_target(TIME, EVENT, h)

    names = list(OOF_BASES.keys())
    P_oof = np.column_stack([OOF_BASES[name][h] for name in names])
    P_test = np.column_stack([TEST_BASES[name][h] for name in names])

    hit_dist = train_fe.loc[(train_raw["event"].astype(int) == 1) & (train_raw["time_to_hit_hours"].astype(float) <= h), "dist_km"].to_numpy(dtype=float)
    if hit_dist.size == 0:
        gate_center = 5.0
    else:
        gate_center = float(np.clip(np.quantile(hit_dist, 0.85), 2.5, 20.0))
    gate_width = float(max(0.75, 0.25 * gate_center + 0.50))

    near_mask = mask & (dist_train <= gate_center)
    far_mask = mask & (dist_train > gate_center)

    near_prior = prior_vector(names, PRIORS[h]["near"])
    far_prior = prior_vector(names, PRIORS[h]["far"])

    if near_mask.sum() < len(names) + 5 or y[near_mask].sum() < 2:
        w_near = near_prior.copy()
    else:
        w_near = optimize_simplex(P_oof[near_mask], y[near_mask], near_prior, BLEND_CFG[h]["reg"], BLEND_CFG[h]["shrink"])

    if far_mask.sum() < len(names) + 5 or y[far_mask].sum() < 2:
        w_far = far_prior.copy()
    else:
        w_far = optimize_simplex(P_oof[far_mask], y[far_mask], far_prior, BLEND_CFG[h]["reg"], BLEND_CFG[h]["shrink"])

    near_pred_oof = np.clip(P_oof @ w_near, 0.0, 1.0)
    near_pred_test = np.clip(P_test @ w_near, 0.0, 1.0)

    far_pred_oof = np.clip(P_oof @ w_far, 0.0, 1.0)
    far_pred_test = np.clip(P_test @ w_far, 0.0, 1.0)

    gate_train = soft_gate(dist_train, gate_center, gate_width)
    gate_holdout = soft_gate(dist_test, gate_center, gate_width)

    raw_oof = gate_train * near_pred_oof + (1.0 - gate_train) * far_pred_oof
    raw_test = gate_holdout * near_pred_test + (1.0 - gate_holdout) * far_pred_test

    cal_oof, cal_test = repeated_isotonic(raw_oof, raw_test, h)

    blend_oof[h] = np.clip(0.85 * cal_oof + 0.15 * raw_oof, 0.0, 1.0)
    blend_test[h] = np.clip(0.85 * cal_test + 0.15 * raw_test, 0.0, 1.0)

    weight_log[h] = {
        "names": names,
        "near_weights": {name: float(val) for name, val in zip(names, w_near)},
        "far_weights": {name: float(val) for name, val in zip(names, w_far)},
        "gate_center_km": gate_center,
        "gate_width_km": gate_width,
    }

pred_oof = np.column_stack(
    [
        blend_oof[12],
        blend_oof[24],
        blend_oof[48],
        np.ones(len(train_raw), dtype=float),
    ]
)
pred_test = np.column_stack(
    [
        blend_test[12],
        blend_test[24],
        blend_test[48],
        np.ones(len(test_raw), dtype=float),
    ]
)

pred_oof = monotone_rows(pred_oof)
pred_test = monotone_rows(pred_test)
pred_oof[:, 3] = 1.0
pred_test[:, 3] = 1.0

score, c_idx, wb, b24, b48, b72 = compute_proxy_hybrid(pred_oof[:, 0], pred_oof[:, 1], pred_oof[:, 2], pred_oof[:, 3])
b12 = compute_brier(TIME, EVENT, pred_oof[:, 0], 12)

print(f"OOF proxy hybrid : {score:.6f}")
print(f"OOF proxy c-index: {c_idx:.6f}")
print(f"OOF weighted brier: {wb:.6f}")
print(f"OOF brier@12: {b12:.6f}")
print(f"OOF brier@24: {b24:.6f}")
print(f"OOF brier@48: {b48:.6f}")
print(f"OOF brier@72: {b72:.6f}")

for h in HORIZONS:
    print(f"h={h} | gate_center={weight_log[h]['gate_center_km']:.3f} km | gate_width={weight_log[h]['gate_width_km']:.3f} km")

submission = sample_sub[["event_id"]].copy()
submission["prob_12h"] = pred_test[:, 0]
submission["prob_24h"] = pred_test[:, 1]
submission["prob_48h"] = pred_test[:, 2]
submission["prob_72h"] = pred_test[:, 3]


def validate_submission(sub, sample):
    required = ["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]
    assert list(sub.columns) == required
    assert len(sub) == len(sample)
    assert sub["event_id"].equals(sample["event_id"])
    assert sub["event_id"].is_unique

    vals = sub[required[1:]].to_numpy(dtype=float)
    assert np.isfinite(vals).all()
    assert ((vals >= -1e-12) & (vals <= 1.0 + 1e-12)).all()
    assert (np.diff(vals, axis=1) >= -1e-12).all()


validate_submission(submission, sample_sub)

submission.to_csv(OUTPUT_PATH, index=False)

oof_dump = pd.DataFrame(
    {
        "event_id": train_raw["event_id"].values,
        "time_to_hit_hours": TIME,
        "event": EVENT,
        "prob_12h": pred_oof[:, 0],
        "prob_24h": pred_oof[:, 1],
        "prob_48h": pred_oof[:, 2],
        "prob_72h": pred_oof[:, 3],
    }
)
oof_dump.to_csv(OOF_PATH, index=False)

with open(WEIGHTS_PATH, "w", encoding="utf-8") as f:
    json.dump(weight_log, f, indent=2)

print(submission.head())
print("saved:", OUTPUT_PATH)
print("saved:", OOF_PATH)
print("saved:", WEIGHTS_PATH)


DATA_DIR: /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26
TRAIN: (221, 37) TEST: (95, 35)
Calibrating anchor / physics signals...
Training direct horizon models...
Training pseudo-value regressors...
Training shared urgency model...
Blending internal model families...
OOF proxy hybrid : 0.964023
OOF proxy c-index: 0.923319
OOF weighted brier: 0.018533
OOF brier@12: 0.064471
OOF brier@24: 0.034339
OOF brier@48: 0.020578
OOF brier@72: 0.000000
h=12 | gate_center=3.262 km | gate_width=1.316 km
h=24 | gate_center=3.258 km | gate_width=1.314 km
h=48 | gate_center=3.235 km | gate_width=1.309 km
   event_id  prob_12h  prob_24h  prob_48h  prob_72h
0  10662602  0.000219  0.000310  0.000310       1.0
1  13353600  0.504863  0.938634  0.996355       1.0
2  13942327  0.000662  0.000692  0.000692       1.0
3  16112781  0.614089  0.939669  0.996545       1.0
4  17132808  0.010419  0.010419  0.010419       1.0
saved: /kaggle/working/submission.csv
saved: /kaggle/working/oof_predictions.csv
sav